In [1]:
import os
import torch
import numpy as np
import urllib.request
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F

from tqdm import tqdm
from torch.utils.data import Dataset, DataLoader

In [2]:
MODELS_DIR = 'models'
os.makedirs(MODELS_DIR, exist_ok=True)

## Download the data

The best place to access books that are no longer under Copyright is [Project Gutenberg](https://www.gutenberg.org/). Today we recommend using [Alice’s Adventures in Wonderland by Lewis Carroll](https://www.gutenberg.org/files/11/11-0.txt) for consistency. Of course you can experiment with other books as well.

In [3]:
data_url = 'https://www.gutenberg.org/files/219/219-0.txt'
fname = 'heart_of_darkness.txt'

if fname not in os.listdir():
    urllib.request.urlretrieve(data_url, fname)

## Load data and create character to integer mappings

- Open the text file, read the data then convert it to lowercase letters.
- Map each character to a respective number. Keep 2 dictionaries in order to have more easily access to the mappings both ways around.
- Transform the data from a list of characters to a list of integers

In [4]:
# Load data
with open(fname, encoding='utf-8') as f:
    text_data = f.read()
text_data = text_data.lower()

# Characters to integers
chars = sorted(list(set(text_data)))
char_to_int = {char: i for i, char in enumerate(chars)}
int_to_char = {i: char for i, char in enumerate(chars)}
int_data = [char_to_int[char] for char in text_data]

n_chars = len(text_data)
n_vocab = len(chars)

print(f"Total Characters: {n_chars}")
print(f"Total Vocab: {n_vocab}")
print(f"Vocabulary: {char_to_int}")

Total Characters: 209997
Total Vocab: 52
Vocabulary: {'\n': 0, ' ': 1, '!': 2, '&': 3, '(': 4, ')': 5, '*': 6, ',': 7, '-': 8, '.': 9, '0': 10, '1': 11, '2': 12, '6': 13, '9': 14, ':': 15, ';': 16, '?': 17, '[': 18, ']': 19, '_': 20, 'a': 21, 'b': 22, 'c': 23, 'd': 24, 'e': 25, 'f': 26, 'g': 27, 'h': 28, 'i': 29, 'j': 30, 'k': 31, 'l': 32, 'm': 33, 'n': 34, 'o': 35, 'p': 36, 'q': 37, 'r': 38, 's': 39, 't': 40, 'u': 41, 'v': 42, 'w': 43, 'x': 44, 'y': 45, 'z': 46, '—': 47, '‘': 48, '’': 49, '“': 50, '”': 51}


## Define the datasets and dataloaders
- We are "thinking" in sequences of 100 characters: 99 characters in the input and 1 in the output.  
E.g. for the sequence *\['h', 'e', 'l', 'l'\]* as input, we will have *\['o'\]* as the expected output.
- Each pair (sample, label) from the training dataset will be composed from a sequence of 99 ints and a single integer label
- We will keep the first 85% sequences as training data and use the remaining for validation

In [5]:
batch_size = 128
seq_length = 100
train_frac = 0.85

class CharDataset(Dataset):
    def __init__(self, data, seq_length):
        self.data = data
        self.seq_length = seq_length

    def __len__(self):
        return len(self.data) - self.seq_length

    def __getitem__(self, idx):
        input_seq = torch.tensor(self.data[idx:idx+self.seq_length-1], dtype=torch.long)
        target = torch.tensor(self.data[idx+self.seq_length-1], dtype=torch.long)
        return input_seq, target

dataset = CharDataset(int_data, seq_length)

train_size = int(train_frac * len(dataset))
val_size = len(dataset) - train_size
train_dataset, val_dataset = torch.utils.data.random_split(dataset, [train_size, val_size])

train_dataloader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
val_dataloader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False)

print(f"Total sequences: {len(dataset)}")
print(f"Training sequences: {len(train_dataset)}")
print(f"Validation sequences: {len(val_dataset)}")
print(f"Number of training batches: {len(train_dataloader)}")
print(f"Number of validation batches: {len(val_dataloader)}")

Total sequences: 209897
Training sequences: 178412
Validation sequences: 31485
Number of training batches: 1394
Number of validation batches: 246


## Define a model with
- An embedding layer with size 32
- Three LSTM layers with a hidden size of 256 and a dropout rate of 20%
- A final linear classification layer

In [6]:
class Model(nn.Module):
    def __init__(self, n_vocab, embed_size, hidden_size, num_layers, dropout_rate):
        super().__init__()
        self.hidden_size = hidden_size
        self.num_layers = num_layers

        self.embedding = nn.Embedding(n_vocab, embed_size)
        self.lstm = nn.LSTM(embed_size, hidden_size, num_layers, dropout=dropout_rate, batch_first=True)
        self.fc = nn.Linear(hidden_size, n_vocab)

    def forward(self, input_seq, hidden_state=None):
        embedded = self.embedding(input_seq) # Output shape: (batch_size, seq_length-1, embed_size)

        output, hidden_state = self.lstm(embedded, hidden_state) # Output shape: (batch_size, seq_length-1, hidden_size)

        output = self.fc(output[:, -1, :]) # Output shape: (batch_size, n_vocab)

        return output, hidden_state

embed_size = 32
hidden_size = 256
num_layers = 3
dropout_rate = 0.2

model = Model(n_vocab, embed_size, hidden_size, num_layers, dropout_rate)
print(model)


Model(
  (embedding): Embedding(52, 32)
  (lstm): LSTM(32, 256, num_layers=3, batch_first=True, dropout=0.2)
  (fc): Linear(in_features=256, out_features=52, bias=True)
)


## Define the training loop and train the model to predict the next character in the sequence

In [ ]:
def train_one_epoch(model, dataloader, criterion, optimizer, device, epoch_num, num_epochs):
    model.train()
    total_train_loss = 0
    for batch_idx, (inputs, targets) in enumerate(tqdm(dataloader, desc=f"Epoch {epoch_num}/{num_epochs} Training")):
        inputs = inputs.to(device)
        targets = targets.to(device)

        outputs, _ = model(inputs)
        loss = criterion(outputs, targets)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        total_train_loss += loss.item()
    return total_train_loss / len(dataloader)


def validate_one_epoch(model, dataloader, criterion, device, epoch_num, num_epochs):
    model.eval()
    total_val_loss = 0
    with torch.no_grad():
        for inputs, targets in tqdm(dataloader, desc=f"Epoch {epoch_num}/{num_epochs} Validation"):
            inputs = inputs.to(device)
            targets = targets.to(device)

            outputs, _ = model(inputs)
            loss = criterion(outputs, targets)
            total_val_loss += loss.item()
    return total_val_loss / len(dataloader)

In [7]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model.to(device)

learning_rate = 0.001
num_epochs = 10

criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=learning_rate)

checkpoint_path = os.path.join(MODELS_DIR, 'model_checkpoint.pth')

start_epoch = 0
best_val_loss = float('inf')

if os.path.exists(checkpoint_path):
    print(f"Loading model from {checkpoint_path}...")
    checkpoint = torch.load(checkpoint_path)
    model.load_state_dict(checkpoint['model_state_dict'])
    optimizer.load_state_dict(checkpoint['optimizer_state_dict'])
    start_epoch = checkpoint['epoch'] + 1
    best_val_loss = checkpoint['best_val_loss']
    print(f"Resuming training from epoch {start_epoch} with best validation loss: {best_val_loss:.4f}")

print("Starting training...")
for epoch in range(start_epoch, num_epochs):
    avg_train_loss = train_one_epoch(model, train_dataloader, criterion, optimizer, device, epoch + 1, num_epochs)
    avg_val_loss = validate_one_epoch(model, val_dataloader, criterion, device, epoch + 1, num_epochs)

    print(f"Epoch [{epoch+1}/{num_epochs}], Train Loss: {avg_train_loss:.4f}, Val Loss: {avg_val_loss:.4f}")

    if avg_val_loss < best_val_loss:
        best_val_loss = avg_val_loss
        print(f"Validation loss improved. Saving model checkpoint to {checkpoint_path}")
        torch.save({
            'epoch': epoch,
            'model_state_dict': model.state_dict(),
            'optimizer_state_dict': optimizer.state_dict(),
            'best_val_loss': best_val_loss,
        }, checkpoint_path)

print("Training complete!")


Starting training...


Epoch 1/10 Validation: 100%|██████████| 246/246 [00:04<00:00, 56.34it/s]


Epoch [1/10], Train Loss: 2.3432, Val Loss: 1.9985
Validation loss improved. Saving model checkpoint to models/model_checkpoint.pth


Epoch 2/10 Validation: 100%|██████████| 246/246 [00:04<00:00, 56.88it/s]


Epoch [2/10], Train Loss: 1.8884, Val Loss: 1.7807
Validation loss improved. Saving model checkpoint to models/model_checkpoint.pth


...

Epoch [8/10], Train Loss: 1.4447, Val Loss: 1.4937
Validation loss improved. Saving model checkpoint to models/model_checkpoint.pth


Epoch 9/10 Validation: 100%|██████████| 246/246 [00:04<00:00, 55.77it/s]


Epoch [9/10], Train Loss: 1.4140, Val Loss: 1.4802
Validation loss improved. Saving model checkpoint to models/model_checkpoint.pth


Epoch 10/10 Validation: 100%|██████████| 246/246 [00:04<00:00, 55.84it/s]

Epoch [10/10], Train Loss: 1.3854, Val Loss: 1.4700
Validation loss improved. Saving model checkpoint to models/model_checkpoint.pth
Training complete!


## Evaluate the model by generating text

- Start with 99 characters (potentially chosen from a text)
- Generate a new character using the trained network
- Repeat the process by appending the generated character and making a prediction for a new one

In [8]:
def text_to_int(text):
    return [char_to_int[char] for char in text]

def int_to_text(ints):
    return ''.join([int_to_char[i] for i in ints])

def generate_text(model, start_string, num_chars_to_generate, temperature=1.0):
    model.eval()
    generated_text = start_string
    input_eval = torch.tensor(text_to_int(start_string)).unsqueeze(0).to(device)
    hidden = None

    print(f"Generating text with initial string: '{start_string}'")

    with torch.no_grad():
        for _ in range(num_chars_to_generate):
            output, hidden = model(input_eval, hidden)

            output_logits = output.squeeze(0).cpu().numpy() / temperature
            probabilities = F.softmax(torch.tensor(output_logits), dim=-1).numpy()

            predicted_int = np.random.choice(len(probabilities), p=probabilities)

            generated_text += int_to_char[predicted_int]
            input_eval = torch.tensor([[predicted_int]]).to(device)

    return generated_text


start_idx = np.random.randint(0, len(text_data) - (seq_length - 1))
start_string_base = text_data[start_idx:start_idx + (seq_length - 1)]


checkpoint_path = os.path.join(MODELS_DIR, 'model_checkpoint.pth')
if os.path.exists(checkpoint_path):
    checkpoint = torch.load(checkpoint_path)
    model.load_state_dict(checkpoint['model_state_dict'])
    print(f"Model loaded from {checkpoint_path} for text generation.")
else:
    print("Warning: No checkpoint found. Using currently trained model state.")


num_generated_chars = 500
generated = generate_text(model, start_string_base, num_generated_chars, temperature=0.8)
print("\nGenerated text:")
print(generated)


Model loaded from models/model_checkpoint.pth for text generation.
Generating text with initial string: ' any time, because one knows that some real
work is done in there, a deuce of a lot of blue, a litt'

Generated text:
 any time, because one knows that some real
work is done in there, a deuce of a lot of blue, a little land. the first anybody below the standy of this many body to gable that do to reperity of a now of the returning, whispers, the other disominable profounding reserved the glance. i was unexpected deaching by you really work at the deserfle of the steamboat the partish
offering now what carried. but back in a lief right out of
the recollow would the fat and left for him like a blinder when so a mine for a reason. ‘who asked to the
considerent of his exclaped support of the and
and he had a se
